# TimeDepBurgers2D
Notebook to implement and test the time-dependent 2D Burgers' equation.  
Author: Alejandro Diaz  
Date: 1/2/2024

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
from dd_nm_rom import env
env.set(
  backend="numpy",
  device="cpu",
  device_idx=0,
  nb_threads=8,
  epsilon=1e-10,
  floatx="float64",
  seed=0
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from IPython.display import Image

In [ ]:
from dd_nm_rom import fom as fom_mod
from dd_nm_rom import fields as fields_mod

In [ ]:
plt.rc('font', size=20)
plt.rcParams['text.usetex'] = False

In [ ]:
# DD Mesh
nx_intr = 48
ny_intr = 48
lx_sub = 0.5
ly_sub = 0.5
x0 = 0.0
y0 = 0.0
n_sub_x = 2
n_sub_y = 2
# Time grid
dt = 0.03
nt = 100
t_lim = [0, dt*nt]
# PDE
viscosity = 1e-3

In [ ]:
mesh = fom_mod.MeshDD(
  nx_intr=nx_intr,
  ny_intr=ny_intr,
  lx_sub=lx_sub,
  ly_sub=ly_sub,
  x0=x0,
  y0=y0,
  n_sub_x=n_sub_x,
  n_sub_y=n_sub_y
)
mesh.build()
X, Y = mesh.grid

In [ ]:
path_to_figs = "./../../../../../run/figures/"
fig_dir = path_to_figs + f'/unsteady/nx_intr_{nx_intr}_ny_intr_{ny_intr}_lx_{lx_sub}_ly_{ly_sub}/fom/'
os.makedirs(fig_dir, exist_ok=True)

# Single sin peak

In [ ]:
fig_dir_i = fig_dir + "/sin_peak_2by2/"
os.makedirs(fig_dir_i, exist_ok=True)

In [ ]:
field = fields_mod.SinPeak(mesh=mesh, mu_lim=[0.9,1.1])
field.set_params(mu=field.sample_design_space())
U0 = field.u()
V0 = field.v()

#### 2D Burgers equation

In [ ]:
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh
)
fom.build(field)

In [ ]:
# plot exact u and v
plt.figure()
# plt.figure(figsize=(12, 4))
# plt.pcolormesh(X, Y, U0, cmap=cm.jet, shading='auto')
plt.pcolormesh(X, Y, U0, cmap='viridis', shading='auto')#, vmin=Z.min(), vmax=Z.max()) 
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$u$')
plt.savefig(fig_dir_i + '/u0.png', bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure()
# plt.figure(figsize=(12, 4))
# plt.pcolormesh(X, Y, V0, cmap=cm.jet, shading='auto')
plt.pcolormesh(X, Y, V0, cmap='viridis', shading='auto')#, vmin=Z.min(), vmax=Z.max()) 
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$v$')
plt.savefig(fig_dir_i + '/v0.png', bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
# Steady
x0 = np.concatenate([U0.reshape(-1), V0.reshape(-1)])
uv, rhs, converged = fom.solve(x0, dt=dt, nt=nt, steady=False, tol=1e-8, maxit=20, stepsize_min=1e-10, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
# frame updater for animation
def update_frame(Xb, Yb, Z, i, label='$u$'):
  plt.clf()
  plt.pcolormesh(Xb, Yb, Z[i], cmap='viridis', shading='auto', vmin=Z.min(), vmax=Z.max()) 
  plt.xlabel('$x$')
  plt.ylabel('$y$')
  cb = plt.colorbar(orientation='vertical', label=label)
  return plt

In [ ]:
# save gifs of u solution
UU = uv["u"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, UU, i, label='$u$'), frames=len(UU), interval=1)
filename = fig_dir_i + '/u_state.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

In [ ]:
# save gifs of u solution
VV = uv["v"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, VV, i, label='$v$'), frames=len(VV), interval=1)
filename = fig_dir_i + '/v_state.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

#### 2D Burgers equation with domain-decomposition

In [ ]:
ddmdl = fom_mod.DDBurgers2D(fom, constraint_type='strong')
ddmdl.build()

In [ ]:
# Steady
uv_dd, lambdas, rhs, converged = ddmdl.solve(
  x0=ddmdl.get_init_sol(x=x0),
  dt=dt,
  nt=nt,
  steady=False,
  tol=1e-8,
  maxit=20,
  stepsize_min=1e-10,
  verbose=True
)
print("RUNTIME:", ddmdl.runtime)

In [ ]:
# save gifs of u solution
UU_dd = uv_dd["res"]["u"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, UU_dd, i, label='$u$'), frames=len(UU_dd), interval=1)
filename = fig_dir_i + '/u_dd_state.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

In [ ]:
# save gifs of u solution
VV_dd = uv_dd["res"]["v"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, VV_dd, i, label='$v$'), frames=len(VV_dd), interval=1)
filename = fig_dir_i + '/v_dd_state_err.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

In [ ]:
# save gifs of u solution
UU_dd_err = np.abs(uv_dd["res"]["u"] - uv["u"])/np.linalg.norm(uv["u"], axis=0, keepdims=True)
UU_dd_err = UU_dd_err.T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, UU_dd_err, i, label='$u$'), frames=len(UU_dd_err), interval=1)
filename = fig_dir_i + '/u_dd_state_err.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

In [ ]:
# save gifs of u solution
VV_dd_err = np.abs(uv_dd["res"]["v"] - uv["v"])#/np.linalg.norm(uv["v"], axis=0, keepdims=True)
VV_dd_err = VV_dd_err.T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, VV_dd_err, i, label='$v$'), frames=len(VV_dd_err), interval=1)
filename = fig_dir_i + '/v_dd_state_err.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

# Multi sin peak

In [ ]:
fig_dir_i = fig_dir + "/sin_multi_peak_2by2/"
os.makedirs(fig_dir_i, exist_ok=True)

In [ ]:
field = fields_mod.SinMultiPeak(mesh=mesh, mu_lim=[0.5,1.5], forced_config=[1,0,0,0])

In [ ]:
field.set_params(mu=field.sample_design_space())
U0 = field.u()
V0 = field.v()

In [ ]:
# plot exact u and v
plt.figure()
# plt.figure(figsize=(12, 4))
# plt.pcolormesh(X, Y, U0, cmap=cm.jet, shading='auto')
plt.pcolormesh(X, Y, U0, cmap='viridis', shading='auto')#, vmin=Z.min(), vmax=Z.max()) 
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$u$')
plt.savefig(fig_dir_i + '/u0.png', bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure()
# plt.figure(figsize=(12, 4))
# plt.pcolormesh(X, Y, V0, cmap=cm.jet, shading='auto')
plt.pcolormesh(X, Y, V0, cmap='viridis', shading='auto')#, vmin=Z.min(), vmax=Z.max()) 
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$v$')
plt.savefig(fig_dir_i + '/v0.png', bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
# Steady
x0 = np.concatenate([U0.reshape(-1), V0.reshape(-1)])
uv, rhs, converged = fom.solve(x0, dt=dt, nt=nt, steady=False, tol=1e-8, maxit=20, stepsize_min=1e-10, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
# save gifs of u solution
UU = uv["u"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, UU, i, label='$u$'), frames=len(UU), interval=1)
filename = fig_dir_i + '/u_state.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

In [ ]:
# save gifs of u solution
VV = uv["v"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(X, Y, VV, i, label='$v$'), frames=len(VV), interval=1)
filename = fig_dir_i + '/v_state.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

In [ ]:
fig_dir_i = fig_dir + "/sin_multi_peak_10by10/"
os.makedirs(fig_dir_i, exist_ok=True)

In [ ]:
mesh_cfg = mesh.get_config()
mesh_cfg["n_sub_x"] = 10
mesh_cfg["n_sub_y"] = 10
mesh_big = fom_mod.MeshDD(**mesh_cfg)
mesh_big.build()
Xb, Yb = mesh_big.grid

In [ ]:
field = fields_mod.SinMultiPeak(mesh_big, mu_lim=[0.5,1.5])

In [ ]:
field.set_params(mu=field.sample_design_space())
U0 = field.u()
V0 = field.v()

In [ ]:
# plot exact u and v
plt.figure()
# plt.figure(figsize=(12, 4))
# plt.pcolormesh(X, Y, U0, cmap=cm.jet, shading='auto')
plt.pcolormesh(Xb, Yb, U0, cmap='viridis', shading='auto')#, vmin=Z.min(), vmax=Z.max()) 
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$u$')
plt.savefig(fig_dir_i + '/u0.png', bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure()
# plt.figure(figsize=(12, 4))
# plt.pcolormesh(X, Y, V0, cmap=cm.jet, shading='auto')
plt.pcolormesh(Xb, Yb, V0, cmap='viridis', shading='auto')#, vmin=Z.min(), vmax=Z.max()) 
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$v$')
plt.savefig(fig_dir_i + '/v0.png', bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh_big
)
fom.build(field)

In [ ]:
# Steady
x0 = np.concatenate([U0.reshape(-1), V0.reshape(-1)])
uv, rhs, converged = fom.solve(x0, dt=dt, nt=nt, steady=False, tol=1e-8, maxit=20, stepsize_min=1e-10, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
# save gifs of u solution
UU = uv["u"].T.reshape(-1, mesh_big.n["y"], mesh_big.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(Xb, Yb, UU, i, label='$u$'), frames=len(UU), interval=1)
filename = fig_dir_i + '/u_state.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)

In [ ]:
# save gifs of u solution
VV = uv["v"].T.reshape(-1, mesh_big.n["y"], mesh_big.n["x"])
fig = plt.figure()
ani = animation.FuncAnimation(fig, lambda i: update_frame(Xb, Yb, VV, i, label='$v$'), frames=len(VV), interval=1)
filename = fig_dir_i + '/v_state.gif'
ani.save(filename, writer='imagemagick', fps=10)
Image(filename)